## SCD Type-2 Implementation for a single table, e.g. dim_customer

In [0]:
use biki;

## sample table: dim_customer

In [0]:
create or replace table dim_customer (
    surrogate_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    name STRING,
    address STRING,
    city STRING,
    start_date DATE DEFAULT CURRENT_DATE,
    end_date DATE DEFAULT DATE '9999-12-31',
    is_current BOOLEAN DEFAULT TRUE,
    record_hash STRING,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
INSERT INTO dim_customer (customer_id, name, address, city, record_hash)
VALUES
('CUST001','Alice Johnson','123 Maple St','Seattle',MD5(CONCAT_WS('|', 'Alice Johnson', '123 Maple St', 'Seattle'))),
('CUST002','Bob Smith','456 Oak Ave','San Francisco',MD5(CONCAT_WS('|', 'Bob Smith', '456 Oak Ave', 'San Francisco')));

## Assumptions
- Table: dim_customer
- Business key: customer_id
- Columns tracked for changes: name, address, city
- Columns for SCD2: start_date, end_date, is_current, record_hash

## Staging - CDC

In [0]:
CREATE OR REPLACE TABLE staging_customer (
  customer_id STRING,
  name STRING,
  address STRING,
  city STRING
);
INSERT INTO staging_customer (customer_id, name, address, city)
SELECT * FROM VALUES
  ('CUST004', 'Dana White', '321 Elm St', 'Austin'),   -- New customer
  ('CUST005', 'Eve Adams', '654 Spruce Ln', 'Boston'), -- New customer
  ('CUST001', 'Alice Johnson', '999 Cedar St', 'Seattle'), -- Update
  ('CUST002', 'Bob Smith', '456 Oak Ave', 'California'); -- Update

In [0]:
select * from dim_customer;

In [0]:
select * from staging_customer;

## SCD Type-2 Implementation

In [0]:
-- Step 1: Identify what needs to happen
with staging_customer as (
  select *, md5(concat_ws('|',name,address,city)) as record_hash from staging_customer
), staging as (
  select customer_id as merge_key, * from staging_customer
  union all
  -- creates the 'insert' records for rows that already exist but have different data (the 'Update' part of the SCD 2)
  select null as merge_key, stg.* from staging_customer as stg inner join dim_customer as cus 
  on cus.is_current=true and stg.customer_id=cus.customer_id and cus.record_hash<>stg.record_hash
) merge into dim_customer as tgt using staging as stg on tgt.customer_id=stg.merge_key and tgt.is_current=true
-- Step 2: Update the old records (Expire them)
when matched and (tgt.record_hash<>stg.record_hash) then
  update set tgt.is_current=false, end_date=current_date()-1, updated_at=current_date()
-- Step 3: Insert the new records (Brand new + New versions of old records)
when not matched then
insert (customer_id, name, address, city, start_date, end_date, is_current, record_hash) 
values (stg.customer_id, stg.name, stg.address, stg.city, current_date(), '9999-12-31', true, stg.record_hash)

In [0]:
select * from dim_customer;